In [1]:
import json
import pandas as pd
from typing import List, Dict, Optional

# List of common ADC champions (based on typical bot lane carries)
ADC_CHAMPIONS = [
    "ashe", "caitlyn", "draven", "ezreal", "jinx", "kaisa", "kalista", "lucian",
    "missfortune", "samira", "sivir", "tristana", "twitch", "vayne", "xayah", "zeri"
]

def load_data(file_path: str) -> List[Dict]:
    """Load champion data from JSON file."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data
    except FileNotFoundError:
        print(f"Error: {file_path} not found.")
        return []
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON format in {file_path}.")
        return []

def clean_win_rate(win_rate: str) -> float:
    """Convert win rate string (e.g., '52.3%') to float (e.g., 52.3)."""
    try:
        return float(win_rate.strip("%"))
    except ValueError:
        return 0.0

def calculate_adc_score(
    adc_data: Dict,
    enemy_team: List[str],
    allied_team: List[str],
    min_games: int = 100,
    matchup_weight: float = 0.6,
    synergy_weight: float = 0.4
) -> Dict:
    """Calculate a score for an ADC based on matchups and synergies."""
    matchup_win_rates = []
    synergy_win_rates = []

    # Process matchups
    for matchup in adc_data.get("matchups", []):
        opponent = matchup.get("opponent", "").lower()
        if opponent in enemy_team:
            try:
                games = int(matchup.get("games", "0").replace(",", ""))
                if games >= min_games:
                    win_rate = clean_win_rate(matchup.get("win_rate", "0%"))
                    matchup_win_rates.append(win_rate)
            except ValueError:
                continue

    # Process synergies
    for synergy in adc_data.get("synergies", []):
        ally = synergy.get("ally", "").lower()
        if ally in allied_team:
            try:
                games = int(synergy.get("games", "0").replace(",", ""))
                if games >= min_games:
                    win_rate = clean_win_rate(synergy.get("win_rate", "0%"))
                    synergy_win_rates.append(win_rate)
            except ValueError:
                continue

    # Calculate average win rates
    avg_matchup_win_rate = sum(matchup_win_rates) / len(matchup_win_rates) if matchup_win_rates else 50.0
    avg_synergy_win_rate = sum(synergy_win_rates) / len(synergy_win_rates) if synergy_win_rates else 50.0

    # Combine scores (default to 50% if no data)
    score = (matchup_weight * avg_matchup_win_rate) + (synergy_weight * avg_synergy_win_rate)

    return {
        "champion": adc_data["champion"],
        "score": score,
        "avg_matchup_win_rate": avg_matchup_win_rate,
        "avg_synergy_win_rate": avg_synergy_win_rate,
        "matchup_details": [
            {"opponent": m["opponent"], "win_rate": m["win_rate"], "games": m["games"]}
            for m in adc_data.get("matchups", []) if m.get("opponent", "").lower() in enemy_team
        ],
        "synergy_details": [
            {"ally": s["ally"], "win_rate": s["win_rate"], "games": s["games"]}
            for s in adc_data.get("synergies", []) if s.get("ally", "").lower() in allied_team
        ]
    }

def recommend_adc(
    enemy_team: List[str],
    allied_team: List[str],
    data_file: str = "lolalytics_champion_data.json",
    min_games: int = 100
) -> Optional[Dict]:
    """Recommend the best ADC based on matchup and synergy data."""
    # Normalize inputs
    enemy_team = [champ.lower() for champ in enemy_team]
    allied_team = [champ.lower() for champ in allied_team]

    # Load data
    data = load_data(data_file)
    if not data:
        return None

    # Filter for ADC champions
    adc_data = [champ for champ in data if champ["champion"].lower() in ADC_CHAMPIONS]
    if not adc_data:
        print("No ADC champions found in data.")
        return None

    # Calculate scores for each ADC
    scores = [
        calculate_adc_score(adc, enemy_team, allied_team, min_games)
        for adc in adc_data
    ]

    # Sort by score (descending)
    scores.sort(key=lambda x: x["score"], reverse=True)

    # Return top pick with details
    if scores:
        top_pick = scores[0]
        print(f"\nRecommended ADC: {top_pick['champion'].capitalize()}")
        print(f"Score: {top_pick['score']:.2f}%")
        print(f"Average Matchup Win Rate: {top_pick['avg_matchup_win_rate']:.2f}%")
        print(f"Average Synergy Win Rate: {top_pick['avg_synergy_win_rate']:.2f}%")
        print("\nMatchup Details:")
        for matchup in top_pick["matchup_details"]:
            print(f"  vs {matchup['opponent']}: {matchup['win_rate']} ({matchup['games']} games)")
        print("\nSynergy Details:")
        for synergy in top_pick["synergy_details"]:
            print(f"  with {synergy['ally']}: {synergy['win_rate']} ({synergy['games']} games)")
        return top_pick
    return None

def main():
    # Example input (replace with your actual team compositions)
    enemy_team = ["zed", "leona", "viktor", "jarvaniv", "jinx"]
    allied_team = ["lux", "amumu", "darius", "akali"]

    print("Enemy Team:", ", ".join(enemy_team))
    print("Allied Team (excluding ADC):", ", ".join(allied_team))
    recommendation = recommend_adc(enemy_team, allied_team)
    if not recommendation:
        print("No recommendation could be made. Check data file or inputs.")

if __name__ == "__main__":
    main()

Enemy Team: zed, leona, viktor, jarvaniv, jinx
Allied Team (excluding ADC): lux, amumu, darius, akali
Error: lolalytics_champion_data.json not found.
No recommendation could be made. Check data file or inputs.
